# RAG 메트릭 데모 — SDK & API

검색 기반 생성(RAG) 에이전트용 메트릭을 SDK와 API 두 방식으로 계산한다.

대상 메트릭: `recall_at_k`, `precision_at_k`, `ndcg_at_k`, `faithfulness`, `consistency`

- 검색 품질(recall/precision/ndcg): `metadata`의 검색 id와 정답 id를 비교한다.
- 근거성(faithfulness/consistency): 응답(`output`)을 검색된 근거(`retrieved_context`)에 대해 심판한다.

## 사전 준비

```bash
uv sync --extra server --extra t2s
```

## 0. 데이터셋

각 항목은 질문, 응답, 검색된 근거 텍스트/ id, 정답(relevant) id, 그리고 NDCG용 등급 관련도(`relevance`)를 갖는다.

In [ ]:
DATASET = [
    {
        "input": "회사의 환불 정책은 어떻게 되나요?",
        "output": "구매 후 30일 이내에는 전액 환불이 가능합니다.",
        "expected": "30일 이내 전액 환불",
        "retrieved_context": ["환불 정책: 제품 구매 후 30일 이내에 요청하면 전액 환불됩니다.", "배송은 보통 3~5일 걸립니다."],
        "retrieved_ids": ["doc1", "doc7", "doc3"],
        "relevant_ids": ["doc1"],
        "relevance": {"doc1": 3},
    },
    {
        "input": "연차 몇 일까지 쓸 수 있나요?",
        "output": "연차는 총 15일이 제공되며 다음 해로 이월할 수 있습니다.",
        "expected": "15일, 이월 가능",
        "retrieved_context": ["연차는 연간 15일 제공됩니다.", "사용하지 않은 연차는 다음 해로 이월됩니다."],
        "retrieved_ids": ["doc5", "doc2", "doc9"],
        "relevant_ids": ["doc2", "doc5"],
        "relevance": {"doc5": 3, "doc2": 2},
    },
    {
        "input": "재택근무 신청은 어떻게 하나요?",
        "output": "재택근무는 팀장 승인 후 인사 시스템에서 신청합니다.",
        "expected": "팀장 승인 후 인사 시스템 신청",
        "retrieved_context": ["재택근무는 팀장 승인이 필요합니다.", "신청은 인사 시스템에서 진행합니다."],
        "retrieved_ids": ["doc8", "doc4"],
        "relevant_ids": ["doc1", "doc4"],
        "relevance": {"doc4": 2, "doc1": 3},
    },
]
for i, row in enumerate(DATASET):
    print(i, row["input"], "| retrieved:", row["retrieved_ids"], "| relevant:", row["relevant_ids"])

---
# Part 1. SDK

In [ ]:
from agent_eval.core.contracts import EvalContext, MetaKey
from agent_eval.judges.backend import FunctionJudge, lexical_overlap_judge
from agent_eval.metrics.rag import Faithfulness, NdcgAtK, PrecisionAtK, RecallAtK, ResponseConsistency

# 데이터셋 전체를 러너로 집계해 대표값과 95% 신뢰구간(CI)을 구하는 헬퍼.
# API 응답의 "aggregate"와 정확히 같은 계산이다(러너 하나가 두 곳에서 재사용된다).
from agent_eval.core.gate import GatePolicy
from agent_eval.core.suite import Suite
from agent_eval.offline.runner import evaluate


def aggregate(metric, ctxs):
    suite = Suite("demo", metric.name, [metric], GatePolicy())
    return evaluate(suite, ctxs).aggregates[0]


def run_sdk(metric, ctxs):
    """각 항목을 개별 채점한 뒤 데이터셋 집계를 출력한다."""
    print(f"[SDK] {metric.name}")
    for i, ctx in enumerate(ctxs):
        r = metric.score(ctx)
        print(f"  #{i}: score={r.score:.3f}  passed={r.passed}  error={r.error}")
    agg = aggregate(metric, ctxs)
    print(f"  ▶ 집계 value={agg.value:.3f}  95% CI=[{agg.ci_low:.3f}, {agg.ci_high:.3f}]  n={agg.n}")

contexts = [
    EvalContext(
        input=row["input"],
        output=row["output"],
        expected=row["expected"],
        retrieved_context=row["retrieved_context"],
        metadata={
            MetaKey.RETRIEVED_IDS: row["retrieved_ids"],
            MetaKey.RELEVANT_IDS: row["relevant_ids"],
            MetaKey.RELEVANCE: row["relevance"],
        },
    )
    for row in DATASET
]
judge = FunctionJudge(lexical_overlap_judge)
K = 3
print("준비 완료:", len(contexts), "개 컨텍스트, k =", K)

## 1-1. `recall_at_k` — 재현율@k

정답(relevant) id 중 상위 k개 검색 결과에 들어온 비율. 검색기가 놓친 정답이 얼마나 되는지.

In [ ]:
run_sdk(RecallAtK(k=K), contexts)

## 1-2. `precision_at_k` — 정밀도@k

상위 k개 검색 결과 중 실제 정답인 비율. 검색 결과의 잡음 정도.

In [ ]:
run_sdk(PrecisionAtK(k=K), contexts)

## 1-3. `ndcg_at_k` — 정규화 DCG@k

정답을 더 위로 올릴수록 높은 점수. `metadata['relevance']`의 등급 관련도를 사용한다.

In [ ]:
run_sdk(NdcgAtK(k=K), contexts)

## 1-4. `faithfulness` — 근거 충실도

응답(`output`)의 모든 주장이 검색된 근거(`retrieved_context`)에서 뒷받침되는지 심판이 판단한다(할루시네이션 방지).

In [ ]:
run_sdk(Faithfulness(judge), contexts)

## 1-5. `consistency` — 근거 일관성

응답이 근거와 **모순되지** 않는지 심판이 판단한다(누락은 무방, 모순만 감점).

In [ ]:
run_sdk(ResponseConsistency(judge), contexts)

---
# Part 2. API

In [ ]:
# API 파트: 백그라운드 스레드에서 실제 FastAPI 서버를 띄우고, httpx로 진짜 HTTP 요청을 보낸다.
# (환경변수를 설정하지 않으면 서버도 SDK와 동일한 오프라인 스텁 심판을 쓴다 → 점수가 일치한다.)
import threading
import time

import httpx
import uvicorn

from agent_eval.server.app import create_app

PORT = 8078
BASE_URL = f"http://127.0.0.1:{PORT}"

if "server" not in globals():
    server = uvicorn.Server(uvicorn.Config(create_app(), host="127.0.0.1", port=PORT, log_level="warning"))
    threading.Thread(target=server.run, daemon=True).start()
    while not server.started:
        time.sleep(0.1)
print("API 서버 준비 완료:", BASE_URL)
print("health:", httpx.get(f"{BASE_URL}/health").json())

def call_api(path, contexts, params=None):
    """엔드포인트에 contexts를 POST하고, 개별 결과와 집계를 출력한다."""
    payload = {"contexts": contexts}
    if params:
        payload["params"] = params
    resp = httpx.post(BASE_URL + path, json=payload)
    print(f"[API] POST {path} → {resp.status_code}")
    data = resp.json()
    if resp.status_code != 200:
        print("  오류:", data.get("detail"))
        return data
    for i, item in enumerate(data["results"]):
        print(f"  #{i}: score={item['score']:.3f}  passed={item['passed']}  error={item['error']}")
    agg = data["aggregate"]
    print(f"  ▶ 집계 value={agg['value']:.3f}  95% CI=[{agg['ci_low']:.3f}, {agg['ci_high']:.3f}]  n={agg['n']}")
    return data

## 2-1. `POST /rag/recall_at_k`

검색 메트릭은 `metadata`의 `retrieved_ids`·`relevant_ids`를 읽고, `params`로 k를 넘긴다.

In [ ]:
ctx = [{"metadata": {"retrieved_ids": r["retrieved_ids"], "relevant_ids": r["relevant_ids"]}} for r in DATASET]
call_api("/rag/recall_at_k", ctx, params={"k": K})

## 2-2. `POST /rag/precision_at_k`

In [ ]:
ctx = [{"metadata": {"retrieved_ids": r["retrieved_ids"], "relevant_ids": r["relevant_ids"]}} for r in DATASET]
call_api("/rag/precision_at_k", ctx, params={"k": K})

## 2-3. `POST /rag/ndcg_at_k`

등급 관련도를 쓰려면 `metadata['relevance']`도 함께 보낸다.

In [ ]:
ctx = [{"metadata": {"retrieved_ids": r["retrieved_ids"], "relevant_ids": r["relevant_ids"], "relevance": r["relevance"]}} for r in DATASET]
call_api("/rag/ndcg_at_k", ctx, params={"k": K})

## 2-4. `POST /rag/faithfulness`

응답(`output`)과 검색된 근거(`retrieved_context`)를 담는다.

In [ ]:
ctx = [{"output": r["output"], "retrieved_context": r["retrieved_context"]} for r in DATASET]
call_api("/rag/faithfulness", ctx)

## 2-5. `POST /rag/consistency`

In [ ]:
ctx = [{"output": r["output"], "retrieved_context": r["retrieved_context"]} for r in DATASET]
call_api("/rag/consistency", ctx)